> ⚠️ First, you need to **select the correct environment/kernel** to run the following notebook. Please select the kernel named **"Python (CSBDeep)"** ⚠️

You should have perform preprocessing of data using Notebook #1 [preprocessing.ipynb](preprocessing.ipynb) and have two stack of images corresponding to high and low SNR training paired data.

# 2D-CARE Training

This notebook performs the complete training pipeline for Content-Aware Image Restoration (Weigert et al, 2018), or CARE model, optimizing it for low SNR SMLM image denoising. 

The neural network at the heart of 2D-CARE is a U-Net based on convolutional neural networks (CNNs). Its "U-shaped" structure is designed for image restoration because it allows for the capture of both the global context (shapes) and local details (texture/pixels):

<figure style="text-align: center;">
    <img src="../images/unet.png" width="600" />
    <figcaption style="font-style: italic; color: gray;">
        U-net architecture (Ronneberger et al., 2015).
    </figcaption>
</figure>

The training pipeline is the same as the original paper:

1. **Data Loading & Preprocessing**: Loading the paired ROI stacks generated by the patch extraction pipeline.
2. **Model Configuration**: Setting up the CARE network architecture and hyperparameters (learning rate, epochs, batch size,...).
3. **Training**: Launching the actual training.

## Importing libraries & functions
In the following section, we will load Python librairies and functions required. Run the next cell to load librairies and dependencies:

In [1]:
# Run to load libraries

import os
import sys

import time as time
import numpy as np

from pathlib import Path
from ipywidgets import widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))

from care.training import do_training
from care.generate_config import generate_training_config
from care.data_augmentation import apply_data_augmentation_to_npz
from care.data_processing import do_data_processing, inspect_channels_in_npz

## Training dataset generation

In this section, we load the two stack of data, and as described [here](https://nbviewer.org/url/csbdeep.bioimagecomputing.com/examples/denoising2D/1_datagen.ipynb), data are processed into a RawData object, required to run with 2D-CARE. 

**For the CARE pipeline to function correctly, the main directory must be named `Training` and contain two subfolders named exactly as follows**:

```text
└─ Training/
    ├─ Low/
    └─ High/
```

**Data must be in stack, no individual frame, and should have the same naming** (*eg: Training/Low/Training.tif and Training/High/Training.tif*)

Below, you can load the two paired stacks and generates 2D-CARE training data:

In [ ]:
# Run to generate training dataset

fc = FileChooser('.')
fc.title = '<b>Select "Training" directory :</b>'
fc.show_only_dirs = True

twindow_low = widgets.IntText(value=3, description='Time window for low images:', style={'description_width': 'initial'})
twindow_high = widgets.IntText(value=3, description='Time window for high images:', style={'description_width': 'initial'})
simu_checkbox = widgets.Checkbox(value=False, description='Simulated data', indent=False)

button_run = widgets.Button(description='Generate Training Data', button_style='success', icon='play')
output = widgets.Output()

def on_button_clicked(b):
    global DATA_PATH, MODEL_NAME, SAVE_IN
    with output:
        output.clear_output()
        if fc.selected is None:
            print("Error : Please select a directory.")
            return
            
        DATA_PATH = fc.selected
        path_obj = Path(DATA_PATH)
        if path_obj.name != 'Training':
            print(f"Warning : Directory is named '{path_obj.name}' instead of 'Training'.")
            
        MODEL_NAME = path_obj.parent.name
        base_save_in = str(path_obj).replace(f'/{path_obj.name}', '/models/')
        SAVE_IN = f"{base_save_in}{MODEL_NAME}"
        
        print("="*60)
        print(f"Path read: {DATA_PATH}")
        print(f"Model: {MODEL_NAME}")
        print(f"Saved in: {SAVE_IN}")
        print(f"Parameters: Time windows=({twindow_low.value}, {twindow_high.value}), Simulation={simu_checkbox.value}")
        print("="*60)
        
        print("Launching...\n")
        do_data_processing(DATA_PATH, SAVE_IN, twindow_low.value, twindow_high.value, simu_checkbox.value)
        print("Finished!")

button_run.on_click(on_button_clicked)
ui = widgets.VBox([fc, widgets.VBox([twindow_low, twindow_high]), simu_checkbox, button_run, output])
display(ui)

## Data augmentation (*optional*)

When data is sparse or limited in quantity, data augmentation can be used to increase the number of training patches. Data augmentation helps reduce overfitting and enhance invariance.

The augmentations consist of **rotations (0°, 90°, 180°, 270°)** and **vertical flips (mirroring)** for each rotation, generating 8 unique versions for each patch. Run the cell below if needed:

In [ ]:
# Run to perform data augmentation

model_path = SAVE_IN + '_model.npz'
apply_data_augmentation_to_npz(model_path)
inspect_channels_in_npz(model_path, np.random.choice(range(0, 100)))

## Launching 2D-CARE Training

To start a training run, the process is the same as for the published version of 2D-CARE. We have made the hyperparameters adjustable (``kernel size``, ``U-net depth``, etc.) so that we can test and customize the network according to our data. By default, the size of the first layer is set to 64, which corresponds to the size of our images (64x64). 

It is also necessary to adjust the ``n_channel_in`` parameter based on the number of channels used to simulate the temporal context. Other parameters can also be adjusted (``batch size``, ``steps per epoch``, etc.) depending on the capabilities of the machine being used. All these parameters are already included in the initial version of 2D-CARE.

**Monitoring**

During training, you can verify that the training is proceeding correctly. First, in addition to the real-time monitoring displayed in the terminal (training and validation loss values, MAE, MSE, etc.), you can use TensorBoard to track loss plots, the images used, and more, as described in the CSBDeep documentation. Once you're in the current directory, run the command ``tensorboard --logdir=.``, and then open http://localhost:6006/ in a browser. 

In [ ]:
# Setup the training
TRAINING_SETTINGS = {}

# wodgets
w_kernel = widgets.IntText(value=3, description='Kernel Size:')
w_depth  = widgets.IntText(value=1, description='Net Depth:')
w_first  = widgets.IntText(value=64, description='First Layer:')
w_epochs = widgets.IntText(value=2, description='Epochs:')
w_lr     = widgets.FloatText(value=1e-4, description='Learn Rate:', step=1e-5)

ui_hyper = widgets.VBox([widgets.HTML("<b>Hyperparameters</b>"), widgets.HBox([w_kernel, w_depth]), widgets.HBox([w_first, w_epochs]), w_lr])

w_loss_choice = widgets.Dropdown(options=['MSE Standard (CSBDeep)', 'Custom SSIM+MSE (Fixed Data)', 'Custom SPT Data'],
                                 value='MSE Standard (CSBDeep)',
                                 description='Loss choice:',
                                 layout={'width': 'max-content'})

w_custom_params = widgets.Checkbox(value=False, description='Change loss parameters', indent=False)
w_gamma = widgets.FloatText(value=2.0, description='Gamma:')
w_alpha = widgets.FloatText(value=0.65, description='Alpha:')
w_spt   = widgets.IntText(value=20, description='SPT Param:')

ui_loss_params = widgets.VBox([]) 

def update_loss_ui(*args):
    if not w_custom_params.value or w_loss_choice.value == 'MSE Standard (CSBDeep)':
        ui_loss_params.children = []
    elif w_loss_choice.value == 'Custom SSIM+MSE (Fixed Data)':
        ui_loss_params.children = [w_gamma, w_alpha]
    elif w_loss_choice.value == 'Custom SPT Data':
        ui_loss_params.children = [w_spt]

w_loss_choice.observe(update_loss_ui, 'value')
w_custom_params.observe(update_loss_ui, 'value')
ui_loss = widgets.VBox([widgets.HTML("<br><b>Loss function</b>"), w_loss_choice, w_custom_params, ui_loss_params])

btn_save = widgets.Button(description="Save config", button_style='primary', layout={'margin': '20px 0px 0px 0px'})
out_save = widgets.Output()

def on_save_clicked(b):
    global TRAINING_SETTINGS
    with out_save:
        clear_output(wait=True)
        loss_params = {'gamma': w_gamma.value, 'alpha': w_alpha.value, 'spt_val': w_spt.value} if w_custom_params.value else None
        config = generate_training_config(DATA_PATH, SAVE_IN, w_kernel.value, w_depth.value, w_first.value, w_epochs.value, w_lr.value)
        TRAINING_SETTINGS = {'config': config, 'loss_choice': w_loss_choice.value, 'loss_params': loss_params}
        
        print("Config saved!")
        print(f"Model : {config['save_in'].split('/')[-1]}")
        print("Training ready to be launch.")

btn_save.on_click(on_save_clicked)
display(ui_hyper, ui_loss, btn_save, out_save)

Button(button_style='primary', description='💾 Sauvegarder la Config', layout=Layout(margin='20px 0px 0px 0px')…

Output()

Then launch the actual training:

In [ ]:
# Launch training
if not TRAINING_SETTINGS:
    print("Error : Please setup a configuration above first.")
else:
    print(f"Launching 2D-CARE training with the following loss : {TRAINING_SETTINGS['loss_choice']}")
    
    c = TRAINING_SETTINGS['config']
        
    do_training(
        kernel_size=c['kernel_size'], 
        unet_depth=c['unet_depth'], 
        unet_first_layer=c['unet_first_layer'], 
        train_epochs=c['train_epochs'], 
        lr=c['lr'], 
        data_care=c['data_care'], 
        save_in=c['save_in'],
        loss_choice=TRAINING_SETTINGS['loss_choice'],
        loss_params=TRAINING_SETTINGS['loss_params']
    )

🔥 Lancement de l'entraînement avec la loss : Custom SSIM+MSE (Fixed Data)
number of training images:	 180
number of validation images:	 20
image size (2D):		 (64, 64)
axes:				 SYXC
channels in / out:		 3 / 3
Config(n_dim=2, axes='YXC', n_channel_in=3, n_channel_out=3, train_checkpoint='weights_best.h5', train_checkpoint_last='weights_last.h5', train_checkpoint_epoch='weights_now.h5', probabilistic=False, unet_residual=False, unet_n_depth=1, unet_kern_size=3, unet_n_first=64, unet_last_activation='linear', unet_input_shape=(None, None, 3), train_loss='mae', train_epochs=2, train_steps_per_epoch=400, train_learning_rate=0.0001, train_batch_size=16, train_tensorboard=True, train_reduce_lr={'factor': 0.5, 'patience': 10, 'min_delta': 0.0001})
Metal device set to: Apple M4

systemMemory: 16.00 GB
maxCacheSize: 5.92 GB

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param 

2026-06-03 16:54:55.849964: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-03 16:54:55.850675: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-06-03 16:54:56.074667: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Epoch 1/2


2026-06-03 16:54:56.583316: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


 56/400 [===>..........................] - ETA: 14s - loss: 0.1792 - mse: 5196.7041 - mae: 51.0748

KeyboardInterrupt: 